# question2
Fetch paginated API data, flatten, enrich, write Delta to /site_info/person_info

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit, current_date, regexp_extract, explode

In [0]:
all_users = []
limit = 10
skip = 0

while True:
    url = f"https://dummyjson.com/users?limit={limit}&skip={skip}"
    response = requests.get(url)
    data = response.json()
    
    users = data.get("users", [])
    
    if not users: 
        break
    
    all_users.extend(users)
    skip += limit
    print(f"Fetched {len(users)} users at skip={skip - limit}, total so far: {len(all_users)}")

print(f"\nTotal users fetched: {len(all_users)}")

In [0]:
address_schema = StructType([
    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("stateCode", StringType(), True),
    StructField("postalCode", StringType(), True),
    StructField("country", StringType(), True),
    StructField("coordinates", StructType([
        StructField("lat", DoubleType(), True),
        StructField("lng", DoubleType(), True)
    ]), True)
])

bank_schema = StructType([
    StructField("cardExpire", StringType(), True),
    StructField("cardNumber", StringType(), True),
    StructField("cardType", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("iban", StringType(), True)
])

company_schema = StructType([
    StructField("department", StringType(), True),
    StructField("name", StringType(), True),
    StructField("title", StringType(), True),
    StructField("address", address_schema, True)
])

custom_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("username", StringType(), True),
    StructField("birthDate", StringType(), True),
    StructField("bloodGroup", StringType(), True),
    StructField("height", DoubleType(), True),
    StructField("weight", DoubleType(), True),
    StructField("eyeColor", StringType(), True),
    StructField("ip", StringType(), True),
    StructField("macAddress", StringType(), True),
    StructField("university", StringType(), True),
    StructField("ein", StringType(), True),
    StructField("ssn", StringType(), True),
    StructField("userAgent", StringType(), True),
    StructField("role", StringType(), True),
    StructField("address", address_schema, True),
    StructField("bank", bank_schema, True),
    StructField("company", company_schema, True)
])

print(" Custom schema defined")

In [0]:
spark = SparkSession.builder.getOrCreate()


df_raw = spark.createDataFrame(all_users, schema=custom_schema)

print(f" DataFrame created with {df_raw.count()} rows")
df_raw.printSchema()

In [0]:
df_flat = df_raw.select(
    col("id"),
    col("firstName"),
    col("lastName"),
    col("age"),
    col("gender"),
    col("email"),
    col("phone"),
    col("username"),
    col("birthDate"),
    col("bloodGroup"),
    col("height"),
    col("weight"),
    col("eyeColor"),
    col("ip"),
    col("macAddress"),
    col("university"),
    col("ein"),
    col("ssn"),
    col("userAgent"),
    col("role"),
    
    col("address.address").alias("address_street"),
    col("address.city").alias("address_city"),
    col("address.state").alias("address_state"),
    col("address.stateCode").alias("address_stateCode"),
    col("address.postalCode").alias("address_postalCode"),
    col("address.country").alias("address_country"),
    col("address.coordinates.lat").alias("address_lat"),
    col("address.coordinates.lng").alias("address_lng"),
    
    col("bank.cardExpire").alias("bank_cardExpire"),
    col("bank.cardNumber").alias("bank_cardNumber"),
    col("bank.cardType").alias("bank_cardType"),
    col("bank.currency").alias("bank_currency"),
    col("bank.iban").alias("bank_iban"),
  
    col("company.department").alias("company_department"),
    col("company.name").alias("company_name"),
    col("company.title").alias("company_title"),
    col("company.address.city").alias("company_city"),
    col("company.address.state").alias("company_state"),
    col("company.address.country").alias("company_country")
)

print("DataFrame flattened")
df_flat.printSchema()

In [0]:
df_final = df_flat \
    .withColumn(
        "site_address",
        regexp_extract(col("email"), r"@(.+)", 1)   # extracts domain after @
    ) \
    .withColumn("load_date", current_date())

print("Derived columns added")
df_final.select("email", "site_address", "load_date").show(5, truncate=False)

In [0]:
db_name = "site_info"
table_name = "person_info"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {db_name}")

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{db_name}.{table_name}")

print(f" Data written as managed Delta table: {db_name}.{table_name}")

In [0]:
db_name = "site_info"
table_name = "person_info"

df_verify = spark.table(f"{db_name}.{table_name}")
print(f" Rows in Delta table: {df_verify.count()}")
df_verify.show(5, truncate=False)